In [1]:
from tools_eg import *

KeyboardInterrupt: 

In [3]:
import os
import pandas as pd
import copernicusmarine
# copernicusmarine.login()  

out_dir = "/Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS/data/SSH/april_nov"
os.makedirs(out_dir, exist_ok=True)

for day in pd.date_range("2026-06-11", "2026-11-09", freq="D"):
    d = day.strftime("%Y-%m-%d")
    fname = f"{d}_cmems_data.nc"

    if os.path.exists(os.path.join(out_dir, fname)):
        continue  # already downloaded, so a rerun picks up where it stopped

    try:
        copernicusmarine.subset(
            dataset_id="cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D",
            variables=["adt", "ugos", "vgos"],
            minimum_longitude=-70, maximum_longitude=45,
            minimum_latitude=-50, maximum_latitude=0,
            start_datetime=f"{d}T00:00:00",
            end_datetime=f"{d}T00:00:00",
            output_directory=out_dir,
            output_filename=fname,
            username="emmagurcan@gmail.com",
            password="Aslan2011Duman2021!"
        )
        print("ok  ", fname)
    except Exception as e:
        print("FAIL", fname, "->", e)

INFO - 2026-11-10T08:11:03Z - Selected dataset version: "202506"
INFO - 2026-11-10T08:11:03Z - Selected dataset part: "default"


ok   2026-06-21_cmems_data.nc


INFO - 2026-11-10T08:11:06Z - Selected dataset version: "202506"
INFO - 2026-11-10T08:11:06Z - Selected dataset part: "default"


ok   2026-06-22_cmems_data.nc


INFO - 2026-11-10T08:11:10Z - Selected dataset version: "202506"
INFO - 2026-11-10T08:11:10Z - Selected dataset part: "default"


ok   2026-06-23_cmems_data.nc


... (274 outputs left out)

INFO - 2026-11-10T08:19:48Z - Selected dataset version: "202506"
INFO - 2026-11-10T08:19:48Z - Selected dataset part: "default"


ok   2026-11-08_cmems_data.nc


INFO - 2026-11-10T08:19:52Z - Selected dataset version: "202506"
INFO - 2026-11-10T08:19:52Z - Selected dataset part: "default"


ok   2026-11-09_cmems_data.nc


In [ ]:
path="/Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS/data/SSH/april_nov/"

In [ ]:
import xarray as xr
import os
ds=xr.open_dataset(path+os.listdir(path)[0])

In [ ]:
ds

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 223, latitude: 400, longitude: 920)
Coordinates:
  * latitude   (latitude) float32 2kB -49.94 -49.81 -49.69 ... -0.1875 -0.0625
  * longitude  (longitude) float32 4kB -69.94 -69.81 -69.69 ... 44.81 44.94
  * time       (time) datetime64[ns] 2kB 2026-04-01 2026-04-02 ... 2026-11-09
Data variables:
    adt        (time, latitude, longitude) float64 657MB ...
    ugos       (time, latitude, longitude) float64 657MB ...
    vgos       (time, latitude, longitude) float64 657MB ...
Attributes: (12/45)
    Conventions:                     CF-1.6
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    cdm_data_type:                   Grid
    comment:                         Sea Surface Height measured by Altimetry...
    contact:                         servicedesk.cmems@mercator-ocean.eu
    creator_email:                   servicedesk.cmems@mercator-ocean.eu
    ...                              ...
    time_coverage_duration:          P1D
    time_coverage_end:               2024-11-30T12:00:00Z
    time_coverage_resolution:        P1D
    time_coverage_start:             2024-11-29T12:00:00Z
    title:                           NRT merged all satellites Global Ocean G...
    copernicusmarine_version:        2.3.0

In [ ]:

BASE = Path("/Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS")
DIR_SADCP = BASE / "data/SADCP"
SADCP_FILES = {"OS150": DIR_SADCP / "WHIRLS_NB150_TOT_osite.nc",   # full cruise (README)
               "OS38": DIR_SADCP / "WHIRLS_BB38_TOT_osite.nc"}
DIR_LADCP = BASE / "data/LADCP/traitement_avec_SADCP"
DIR_SSH = BASE / "data/SSH"
DIR_EDDIES = {"TOEDDIES_better_ADT": BASE / "data/TOEDDIES/TOEDDIES_better_ADT",
              "TOEDDIES": BASE / "data/TOEDDIES/TOEDDIES"}
FNAME_BATHY = BASE / "data/bathymetry/etopo2.nc"
DIR_TRACKS = BASE / "figures/eddy_tracks"
DIR_CODE = BASE / "analysis"          # where ladcp_tools.py lives

DIR_DRIFTERS =  BASE / "data/DRIFTERS/drifters"
DIR_GEOJSON = BASE / "data/DRIFTERS/drifters_tracks.geojson"

DIR_MVP = BASE / "data/MVP_Data_processed"

In [ ]:
os.listdir(DIR_SADCP)

['WHIRLS_BB38_TOT_osite.nc',
 'MD2_OS38_WHIRLS-2026_120s_oceansites.nc',
 'MD2_OS150_WHIRLS-2026_120s_oceansites.nc',
 'WHIRLS_NB150_TOT_osite.nc',
 'README.txt']

In [ ]:
d = load_sadcp_raw(DIR_SADCP / "WHIRLS_BB38_TOT_osite.nc").resample(time=SADCP_AVG).mean()

  WHIRLS_BB38_TOT_osite.nc: using UVEL_ADCP_CORTIDE, VVEL_ADCP_CORTIDE


In [ ]:
def sadcp_window(inst, sadcp=None, t0=None, t1=None, files=None):
    """OS150 above Z_SPLIT + OS38 below, averaged over SADCP_AVG.
    t0, t1 : optional time limits; None = from the start / to the end of the record.
    files  : dict {"OS150": path, "OS38": path}; default SADCP_FILES.
    Returns a dict of numpy arrays: time, lon, lat, z, u (nt, nz), v (nt, nz)."""
    files = SADCP_FILES if files is None else files
    parts = []
    if sadcp is None:
        for inst, f in files.items():
            if not Path(f).exists():
                print(f"  ! missing {f}")
                continue
            d = load_sadcp_raw(f).sel(time=slice(t0, t1)).resample(time=SADCP_AVG).mean()
            zz = d.z.values
            keep = zz <= Z_SPLIT if inst == "OS150" else (zz > Z_SPLIT) & (zz <= ZMAX_SADCP)
            parts.append(d.isel(z=np.where(keep)[0]))
        if not parts:
            raise FileNotFoundError("no SADCP file found")
    else:
        print("Calculating Mean")
        zz = sadcp.z.values
        keep = zz <= Z_SPLIT if inst == "OS150" else (zz > Z_SPLIT) & (zz <= ZMAX_SADCP)
        parts = [sadcp.isel(z=np.where(keep)[0])]
    out = parts[0]
    for p in parts[1:]:
        # outer join: keep times where either instrument has data
        out, p = xr.align(out, p, join="outer", exclude=["z"])
        out = xr.Dataset({"u": xr.concat([out.u, p.u], "z"), "v": xr.concat([out.v, p.v], "z"),
                          "lat": out.lat.fillna(p.lat), "lon": out.lon.fillna(p.lon)})
    out = out.sortby("z")
    ok = np.isfinite(out.lat.values) & np.isfinite(out.lon.values) & np.isfinite(out.u.values).any(1)
    out = out.isel(time=ok)
    return dict(time=out.time.values, lon=out.lon.values, lat=out.lat.values, z=out.z.values,
                u=out.u.values, v=out.v.values)

In [ ]:
sadcp

{'time': array(['2026-06-29T12:00:00.000000000', '2026-06-29T12:10:00.000000000',
        '2026-06-29T12:20:00.000000000', ...,
        '2026-07-27T15:00:00.000000000', '2026-07-27T15:10:00.000000000',
        '2026-07-27T15:20:00.000000000'], dtype='datetime64[ns]'),
 'lon': array([17.5897131 , 17.57132721, 17.55033493, ..., 17.98534737,
        18.01300125, 18.04140472]),
 'lat': array([-34.5002718 , -34.50020828, -34.50010071, ..., -34.15181122,
        -34.13793106, -34.12350159]),
 'z': array([167.46000671, 191.46000671, 215.46000671, 239.46000671,
        263.45999146, 287.45999146, 311.45999146, 335.45999146,
        359.45999146, 383.45999146, 407.45999146, 431.45999146,
        455.45999146, 479.45999146, 503.45999146, 527.46002197,
        551.46002197, 575.46002197, 599.46002197, 623.46002197,
        647.46002197, 671.46002197, 695.46002197, 719.46002197,
        743.46002197, 767.46002197, 791.46002197, 815.46002197,
        839.46002197, 863.46002197, 887.46002197, 911.46

In [ ]:
zz = d.z.values

In [ ]:
zz

array([  47.45999908,   71.45999908,   95.45999908,  119.45999908,
        143.46000671,  167.46000671,  191.46000671,  215.46000671,
        239.46000671,  263.45999146,  287.45999146,  311.45999146,
        335.45999146,  359.45999146,  383.45999146,  407.45999146,
        431.45999146,  455.45999146,  479.45999146,  503.45999146,
        527.46002197,  551.46002197,  575.46002197,  599.46002197,
        623.46002197,  647.46002197,  671.46002197,  695.46002197,
        719.46002197,  743.46002197,  767.46002197,  791.46002197,
        815.46002197,  839.46002197,  863.46002197,  887.46002197,
        911.46002197,  935.46002197,  959.46002197,  983.46002197,
       1007.46002197, 1031.45996094, 1055.45996094, 1079.45996094,
       1103.45996094, 1127.45996094, 1151.45996094, 1175.45996094,
       1199.45996094, 1223.45996094])

In [ ]:
keep = zz <= Z_SPLIT if inst == "OS150" else (zz > Z_SPLIT) & (zz <= ZMAX_SADCP)

NameError: name 'inst' is not defined